# Challenge 2: entrenamiento y selección del modelo

En esta etapa debes construir pipelines reproducibles, comparar modelos y seleccionar un campeón sin utilizar el conjunto de test.

**Entradas:** `train.csv` y `data_contract.json`  
**Salidas:** `champion_model.joblib`, `training_metadata.json` y `cv_results.csv`

## Reglas

- El preprocesamiento debe estar dentro de cada `Pipeline`.
- La selección debe hacerse con validación cruzada.
- No cargues `test.csv` en este notebook.

In [1]:
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GridSearchCV, StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
from pathlib import Path

PROJECT_DIR = Path.cwd()
RAW_DATA_DIR = PROJECT_DIR / "data" / "raw"
PROCESSED_DATA_DIR = PROJECT_DIR / "data" / "processed"
ARTIFACTS_DIR = PROJECT_DIR / "artifacts"
REPORTS_DIR = PROJECT_DIR / "reports"

for directory in (PROCESSED_DATA_DIR, ARTIFACTS_DIR, REPORTS_DIR):
    directory.mkdir(parents=True, exist_ok=True)

RANDOM_STATE = 42

## 1. Carga de artefactos

In [3]:
train_df = pd.read_csv(PROCESSED_DATA_DIR / "train.csv")
contract = json.loads((ARTIFACTS_DIR / "data_contract.json").read_text(encoding="utf-8"))

X_train = train_df[contract["features"]]
y_train = train_df[contract["target"]]

print(X_train.shape, y_train.shape)

(142, 13) (142,)


## 2. Pipeline de procesamiento

In [4]:
feature_columns = contract["features"]

tree_preprocessor = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median"))]), feature_columns),
])
scaled_preprocessor = ColumnTransformer([
    ("numeric", Pipeline([("imputer", SimpleImputer(strategy="median")),
                          ("scaler", StandardScaler())]), feature_columns),
])

**Pregunta:** ¿por qué el `StandardScaler` debe estar dentro del pipeline y no ajustarse antes de la validación cruzada?

R/ Si el StandardScaler se ajusta antes de la validación cruzada, calcula media y desviación usando también las filas del fold de validación. Eso es fuga de datos (data leakage) y la métrica sale optimista. Dentro del Pipeline, el scaler se reajusta en cada fold solo con sus datos de entrenamiento.

## 3. Modelos candidatos

In [5]:
models = {
    "decision_tree": Pipeline([("preprocessing", clone(tree_preprocessor)),
                               ("classifier", DecisionTreeClassifier(random_state=RANDOM_STATE))]),
    "logistic_regression": Pipeline([("preprocessing", clone(scaled_preprocessor)),
                                     ("classifier", LogisticRegression(max_iter=5000, random_state=RANDOM_STATE))]),
    "knn": Pipeline([("preprocessing", clone(scaled_preprocessor)),
                     ("classifier", KNeighborsClassifier())]),
}

## 4. Comparación inicial con validación cruzada

In [6]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
rows = []
for name, pipe in models.items():
    res = cross_validate(pipe, X_train, y_train, cv=cv, scoring="accuracy",
                         return_train_score=True, n_jobs=-1)
    rows.append({"model": name,
                 "train_accuracy": res["train_score"].mean(),
                 "val_accuracy": res["test_score"].mean(),
                 "val_std": res["test_score"].std()})
initial_comparison = pd.DataFrame(rows).sort_values("val_accuracy", ascending=False).reset_index(drop=True)
initial_comparison

,model,train_accuracy,val_accuracy,val_std
0,logistic_regression,1.0000,0.979064,0.027713
1,knn,0.9771,0.950739,0.035274
2,decision_tree,1.0000,0.902217,0.058617


## 5. Optimización de hiperparámetros

In [7]:
param_grids = {
    "decision_tree": {"classifier__criterion": ["gini", "entropy"],
                      "classifier__max_depth": [2, 3, 4, 5, None],
                      "classifier__min_samples_split": [2, 5, 10]},
    "logistic_regression": {"classifier__C": [0.01, 0.1, 1, 10, 100]},
    "knn": {"classifier__n_neighbors": [3, 5, 7, 9, 11],
            "classifier__weights": ["uniform", "distance"]},
}

searches, rows = {}, []
for name, pipe in models.items():
    s = GridSearchCV(pipe, param_grids[name], scoring="accuracy", cv=cv,
                     n_jobs=-1, return_train_score=True)
    s.fit(X_train, y_train)
    searches[name] = s
    i = s.best_index_
    rows.append({"model": name,
                 "cv_accuracy": s.best_score_,
                 "cv_std": s.cv_results_["std_test_score"][i],
                 "train_accuracy": s.cv_results_["mean_train_score"][i],
                 "best_params": json.dumps(s.best_params_, default=str)})
tuned_comparison = pd.DataFrame(rows).sort_values("cv_accuracy", ascending=False).reset_index(drop=True)
tuned_comparison

,model,cv_accuracy,cv_std,train_accuracy,best_params
0,logistic_regression,0.979064,0.027713,1.000000,"{""classifier__C"": 1}"
1,knn,0.957882,0.040520,0.964773,"{""classifier__n_neighbors"": 7, ""classifier__we..."
2,decision_tree,0.902217,0.058617,0.998246,"{""classifier__criterion"": ""gini"", ""classifier_..."


## 6. Selección y persistencia del campeón

In [8]:
champion_name = tuned_comparison.loc[0, "model"]
champion = searches[champion_name].best_estimator_

joblib.dump(champion, ARTIFACTS_DIR / "champion_model.joblib")

training_metadata = {
    "champion": champion_name,
    "selection_metric": "accuracy",
    "cv_folds": 5,
    "cv_score": float(searches[champion_name].best_score_),
    "best_params": searches[champion_name].best_params_,
}
(ARTIFACTS_DIR / "training_metadata.json").write_text(
    json.dumps(training_metadata, indent=2, default=str), encoding="utf-8")

pd.concat([initial_comparison.assign(stage="initial"),
           tuned_comparison.assign(stage="tuned")]).to_csv(REPORTS_DIR / "cv_results.csv", index=False)
print("Campeón:", champion_name, "| CV acc:", round(training_metadata["cv_score"], 4))

Campeón: logistic_regression | CV acc: 0.9791


In [ ]:
assert (ARTIFACTS_DIR / "champion_model.joblib").exists()
assert (ARTIFACTS_DIR / "training_metadata.json").exists()
assert (REPORTS_DIR / "cv_results.csv").exists()
print("Challenge 2 completado.")

**Justificación de la selección:**

> Explica la decisión considerando desempeño, variabilidad, sobreajuste, complejidad e interpretabilidad.

R/ El campeón es **regresión logística** (C=1), con la mayor accuracy en validación cruzada (0.979) y la menor variabilidad entre folds (desv. 0.028).

- **Desempeño:** supera a KNN (0.958) por ~2 puntos y al árbol de decisión (0.902) por ~8. Frente a KNN la diferencia es menor que una desviación estándar de los folds, así que no es una ventaja contundente, pero va en la misma dirección en media y en estabilidad.
- **Variabilidad:** el árbol tiene la mayor desviación (0.059), lo que indica que su desempeño depende mucho de qué datos caen en cada fold. KNN (0.041) es intermedio y la regresión logística es la más estable.
- **Sobreajuste:** el árbol es el caso más claro: accuracy de entrenamiento 0.998 frente a 0.902 en validación (brecha de ~10 puntos), es decir, memoriza el entrenamiento. La regresión logística llega a 1.000 en train frente a 0.979 en validación (brecha de ~2 puntos), un sobreajuste leve y aceptable. KNN tiene la brecha más pequeña (~0.7 puntos), pero a costa de menor desempeño.
- **Complejidad e interpretabilidad:** la regresión logística es un modelo lineal con un solo hiperparámetro, rápido de entrenar y de aplicar, y sus coeficientes permiten ver qué variables empujan hacia cada clase. Esto debe hacerse con cuidado porque varias variables están muy correlacionadas (por ejemplo `flavanoids` y `total_phenols`). El árbol es el más fácil de leer, pero inestable, y KNN guarda todo el entrenamiento y es costoso en inferencia.
- **Hiperparámetro:** C=1 queda en el centro de la rejilla (0.01 a 100), no en un extremo, así que la búsqueda no parece limitada por el rango explorado.

Se selecciona la regresión logística por combinar el mayor desempeño, la mayor estabilidad y un sobreajuste controlado con un modelo simple. La elección se hizo solo con validación cruzada sobre train; el test no se ha usado.

## Checklist

- [ ] Todo el procesamiento está dentro de pipelines.
- [ ] Comparé tres familias de modelos.
- [ ] Optimicé hiperparámetros con CV.
- [ ] Elegí el campeón sin usar test.
- [ ] Guardé el pipeline completo y sus metadatos.